# **RT-Coder for training(先不用看這個）**

In [3]:
import json
from datasets import Dataset

# jsonl 格式：每行一筆
with open("C:\\Users\\Aries\\OneDrive\\桌面\\Project\\Resyn27k.json", "r") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"訓練筆數：{len(raw_data)}")
print(raw_data[0].keys())
print("---")
print(raw_data[0]["Instruction"][:100])
print("---")
print(raw_data[0]["Response"])

訓練筆數：26532
dict_keys(['Instruction', 'Response'])
---

You are tasked with designing a module for a simple calculator that can perform basic arithmetic op
---
['module calculator(\n    input logic [31:0] a,\n    input logic [31:0] b,\n    input logic reset_n,\n    output logic [31:0] add,\n    output logic [31:0] sub,\n    output logic [31:0] mul,\n    output logic [31:0] div\n);\n\n    always @(a, b, reset_n) begin\n        if(!reset_n) begin\n            add <= 0;\n            sub <= 0;\n            mul <= 0;\n            div <= 0;\n        end else begin\n            add <= a + b;\n            sub <= a - b;\n            mul <= a * b;\n            if(b != 0) begin\n                div <= a / b;\n            end else begin\n                div <= 0;\n            end\n        end\n    end\n\nendmodule']


In [4]:
def format_prompt(sample):
    # user_prompt = f"{sample['instruction']}\n\n{sample['input']}".strip()
    instruction = sample["Instruction"]
    response = sample["Response"]
    if isinstance(response, list):
        response = response[0]

    response = response.replace("\\n", "\n").replace("\\t", "\t")
    
    return {
        "text": f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert RTL/Verilog hardware designer.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction.strip()}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{response.strip()}<|eot_id|>"""
    }

dataset = Dataset.from_list(raw_data).map(format_prompt)
# 只用 5000 筆，訓練約 2~3 小時
dataset = dataset.select(range(6000))

print(f"完成！共 {len(dataset)} 筆")
print(dataset[0]["text"])  # 確認格式
import numpy as np
lengths = [len(d["text"].split()) for d in dataset]
print(f"平均長度: {np.mean(lengths):.0f} tokens")
print(f"最大長度: {np.max(lengths)} tokens")

Map:   0%|          | 0/26532 [00:00<?, ? examples/s]

完成！共 6000 筆
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert RTL/Verilog hardware designer.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
You are tasked with designing a module for a simple calculator that can perform basic arithmetic operations. The module should have two inputs, `a` and `b`, and four outputs, `add`, `sub`, `mul`, and `div`. The module should be able to perform the following operations:
- `add`: add `a` and `b` and output the result
- `sub`: subtract `b` from `a` and output the result
- `mul`: multiply `a` and `b` and output the result
- `div`: divide `a` by `b` and output the result

If `b` is zero, the `div` output should be zero. The module should also have a reset input, `reset_n`, which should reset all outputs to zero when active low.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
module calculator(
    input logic [31:0] a,
    input logic [31:0] b,
    input logic reset_n,
    output logic [31:0] add,
    output logic

In [5]:
import torch
print(torch.cuda.is_available())       # True
print(torch.cuda.get_device_name(0))   # RTX 5070 Ti

True
NVIDIA GeForce RTX 5070 Ti


In [3]:
from huggingface_hub import login
login(token="")

In [95]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     dtype=torch.float16,
#     device_map="auto",
# )
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': torch.cuda.current_device()},
)
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

C:\Users\Aries\anaconda3\envs\Rays_Project\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [5]:
# 載入後先看記憶體
allocated = torch.cuda.memory_allocated() / 1024**3
print(f"模型載入後：{allocated:.2f}GB")
model.save_pretrained(".\\meta-llama3.2-3b-local")
tokenizer.save_pretrained(".\\meta-llama3.2-3b-local")

模型載入後：4.97GB


('.\\meta-llama3.2-3b-local\\tokenizer_config.json',
 '.\\meta-llama3.2-3b-local\\special_tokens_map.json',
 '.\\meta-llama3.2-3b-local\\tokenizer.json')

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
#model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 預期輸出：trainable params: ~20M || all params: ~3B || trainable%: ~0.6%

trainable params: 48,627,712 || all params: 3,261,377,536 || trainable%: 1.4910


In [26]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./llama3.2-3b",
    num_train_epochs=3,
    per_device_train_batch_size=2,      # 從 2 改 4
    gradient_accumulation_steps=4,      # 從 8 改 4
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_torch",                # 拿掉 paged，fp16 不需要
    max_seq_length=2048,                 # 從預設降到 512，速度快很多
    # gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    formatting_func=lambda x: x["text"],
)

trainer.train()

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

C:\Users\Aries\anaconda3\envs\Rays_Project\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,5.113200
20,4.044800
30,3.103000
40,2.744300
50,2.657600
60,2.673100
70,2.700700
80,2.718500
90,2.489200
100,2.444400


C:\Users\Aries\anaconda3\envs\Rays_Project\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\Aries\anaconda3\envs\Rays_Project\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  ret

TrainOutput(global_step=2250, training_loss=1.8553730795118544, metrics={'train_runtime': 11444.5328, 'train_samples_per_second': 1.573, 'train_steps_per_second': 0.197, 'total_flos': 2.4947161251336192e+17, 'train_loss': 1.8553730795118544, 'epoch': 3.0})

In [27]:
# 儲存 LoRA adapter（輕量，幾百MB）
trainer.save_model("C:\\Users\\Aries\\OneDrive\\桌面\\Project\\llama3.2-3b-verilog-adapter-v2")
tokenizer.save_pretrained("C:\\Users\\Aries\\OneDrive\\桌面\\Project\\llama3.2-3b-verilog-adapter-v2")

print("儲存完成！")

儲存完成！


In [3]:
import json
import os
import tempfile
import subprocess
import torch
import pandas as pd
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import re

# ========== 1. 載入 RTLLM 驗證集 ==========
with open(".\\rtllm_sft.jsonl", "r", encoding="utf-8") as f:
    rtllm_data = [json.loads(line) for line in f if line.strip()]

print(f"RTLLM 筆數：{len(rtllm_data)}")

# ========== 2. 安裝 iverilog ==========
#!apt-get install -y iverilog -q

# ========== 3. 評估函式 ==========
def extract_first_module(code):
    """只取第一個完整的 module...endmodule"""
    match = re.search(r'(module\s+\w+.*?endmodule)', code, re.DOTALL)
    if match:
        return match.group(1)
    return code

def generate_verilog(model, tokenizer, sample):
    user_prompt = sample['input']
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert RTL/Verilog hardware designer.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    torch.manual_seed(42)       # ✅
    torch.cuda.manual_seed(42)  # ✅
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    # ✅ 加這行，只取第一個 module
    return extract_first_module(generated.strip())

# def check_syntax(verilog_code):
#     """用 iverilog 檢查語法"""
#     with open("/tmp/test.v", "w") as f:
#         f.write(verilog_code)
#     result = subprocess.run(
#         ["iverilog", "-o", "/tmp/test.out", "/tmp/test.v"],
#         capture_output=True, text=True
#     )
#     return result.returncode == 0

def check_syntax(verilog_code):
    """用 iverilog 檢查語法"""
    # 用 Windows 相容的暫存路徑
    tmp_dir = tempfile.gettempdir()
    tmp_v = os.path.join(tmp_dir, "test.v")
    tmp_out = os.path.join(tmp_dir, "test.out")
    
    with open(tmp_v, "w") as f:
        f.write(verilog_code)
    
    result = subprocess.run(
        ["iverilog", "-o", tmp_out, tmp_v],
        capture_output=True, text=True
    )
    return result.returncode == 0


def compute_bleu(reference, hypothesis):
    """計算 BLEU score"""
    smoothie = SmoothingFunction().method4
    return sentence_bleu(
        [reference.split()],
        hypothesis.split(),
        smoothing_function=smoothie
    )

def evaluate_dataset(model, tokenizer, data, dataset_name):
    """對整個資料集跑評估"""
    print(f"\n{'='*40}")
    print(f"評估中：{dataset_name}（共 {len(data)} 筆）")
    print(f"{'='*40}")

    results = []
    for i, sample in enumerate(tqdm(data)):
        generated = generate_verilog(model, tokenizer, sample)
        reference = sample["output"].strip()

        syntax_ok = check_syntax(generated)
        bleu = compute_bleu(reference, generated)

        results.append({
            "id": i,
            "syntax_pass": syntax_ok,
            
            "bleu": bleu,
            "generated": generated,
            "reference": reference,
        })

    return results

# ========== 4. 跑評估 ==========
# 訓練前（用 base model 評估）
print("載入 base model 評估...")
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 指定本地路徑
# LOCAL_PATH = r"C:\\Users\\Aries\\OneDrive\\桌面\\Project\\meta-llama3.2-3b-local"

# base_model = AutoModelForCausalLM.from_pretrained(
#     LOCAL_PATH,
#     torch_dtype=torch.float16,
#     device_map={'': torch.cuda.current_device()},
# )
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH)

# before_results = evaluate_dataset(base_model, tokenizer, rtllm_data, "Base Model on RTLLM")

#模型重新從huggingface抓才會確保乾淨
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={'': torch.cuda.current_device()},
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

before_results = evaluate_dataset(base_model, tokenizer, rtllm_data, "Base Model on RTLLM")

RTLLM 筆數：50
載入 base model 評估...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


評估中：Base Model on RTLLM（共 50 筆）


100%|██████████| 50/50 [09:59<00:00, 12.00s/it]


In [5]:
# 訓練後（載入 fine-tuned model）
from peft import PeftModel
print("\n載入 fine-tuned model 評估...")
ft_model = PeftModel.from_pretrained(base_model, ".\\llama3.2-3b-verilog-adapter-v2")
ft_model = ft_model.merge_and_unload()
ft_model.eval()

after_results = evaluate_dataset(ft_model, tokenizer, rtllm_data, "Fine-tuned Model on RTLLM")

# ========== 5. 彙整結果表格 ==========
def summarize(results, label):
    syntax_rate = sum(r["syntax_pass"] for r in results) / len(results)
    avg_bleu = sum(r["bleu"] for r in results) / len(results)
    return {
        "Model": label,
        "Syntax Pass Rate": f"{syntax_rate:.2%}",
        "Avg BLEU Score": f"{avg_bleu:.4f}",
        "Syntax Pass (n)": f"{sum(r['syntax_pass'] for r in results)}/{len(results)}",
    }

summary = pd.DataFrame([
    summarize(before_results, "llama3.2-3b Base"),
    summarize(after_results, "llama3.2-3b Fine-tuned"),
])

print("\n")
print("=" * 55)
print("📊 評估結果比較")
print("=" * 55)
print(summary.to_string(index=False))
print("=" * 55)

# ========== 6. 儲存詳細結果 ==========
detail_df = pd.DataFrame([
    {
        "id": r["id"],
        "before_syntax": before_results[r["id"]]["syntax_pass"],
        "after_syntax": after_results[r["id"]]["syntax_pass"],
        "before_bleu": f"{before_results[r['id']]['bleu']:.4f}",
        "after_bleu": f"{after_results[r['id']]['bleu']:.4f}",
    }
    for r in after_results
])

detail_df.to_csv(".\\Llama_eval_detail-celine-v2test.csv", index=False)
print("\n詳細結果已儲存至 Llama_eval_detail-celine-v2test.csv")


載入 fine-tuned model 評估...

評估中：Fine-tuned Model on RTLLM（共 50 筆）


100%|██████████| 50/50 [10:38<00:00, 12.76s/it]



📊 評估結果比較
                 Model Syntax Pass Rate Avg BLEU Score Syntax Pass (n)
      llama3.2-3b Base           32.00%         0.1757           16/50
llama3.2-3b Fine-tuned           28.00%         0.1058           14/50

詳細結果已儲存至 Llama_eval_detail-celine-v2test.csv


<h2>用eval_verilog_sft評估</h2>

In [10]:
# ========== 1. 載入 eval_verilog_sft 驗證集 ==========
with open(".\\eval_verilog_sft.jsonl", "r", encoding="utf-8") as f:
    eval_verilog_sft_data = [json.loads(line) for line in f if line.strip()]

print(f"eval_verilog_sft 筆數：{len(eval_verilog_sft_data)}")

eval_verilog_sft 筆數：156


In [11]:
#模型重新從huggingface抓才會確保乾淨
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={'': torch.cuda.current_device()},
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

before_results = evaluate_dataset(base_model, tokenizer, eval_verilog_sft_data, "Base Model on eval_verilog_sft_data")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


評估中：Base Model on eval_verilog_sft_data（共 156 筆）


100%|██████████| 156/156 [2:21:36<00:00, 54.47s/it]


In [12]:
# 訓練後（載入 fine-tuned model）
from peft import PeftModel
print("\n載入 fine-tuned model 評估...")
ft_model = PeftModel.from_pretrained(base_model, ".\\llama3.2-3b-verilog-adapter")
ft_model = ft_model.merge_and_unload()
ft_model.eval()

after_results = evaluate_dataset(ft_model, tokenizer, eval_verilog_sft_data, "Fine-tuned Model on eval_verilog_sft_data")

summary = pd.DataFrame([
    summarize(before_results, "llama3.2-3b Base on eval_verilog_sft_data"),
    summarize(after_results, "llama3.2-3b Fine-tuned on eval_verilog_sft_data"),
])

print("\n")
print("=" * 55)
print("📊 評估結果比較")
print("=" * 55)
print(summary.to_string(index=False))
print("=" * 55)

# ========== 6. 儲存詳細結果 ==========
detail_df = pd.DataFrame([
    {
        "id": r["id"],
        "before_syntax": before_results[r["id"]]["syntax_pass"],
        "after_syntax": after_results[r["id"]]["syntax_pass"],
        "before_bleu": f"{before_results[r['id']]['bleu']:.4f}",
        "after_bleu": f"{after_results[r['id']]['bleu']:.4f}",
    }
    for r in after_results
])

detail_df.to_csv(".\\eval_verilog_sft_data_Llama_eval_detail-celine-v1test.csv", index=False)
print("\n詳細結果已儲存至 eval_verilog_sft_data_Llama_eval_detail-celine-v2test.csv")


載入 fine-tuned model 評估...

評估中：Fine-tuned Model on eval_verilog_sft_data（共 156 筆）


100%|██████████| 156/156 [3:13:37<00:00, 74.47s/it]



📊 評估結果比較
                                          Model Syntax Pass Rate Avg BLEU Score Syntax Pass (n)
      llama3.2-3b Base on eval_verilog_sft_data           33.97%         0.1783          53/156
llama3.2-3b Fine-tuned on eval_verilog_sft_data           74.36%         0.3046         116/156

詳細結果已儲存至 eval_verilog_sft_data_Llama_eval_detail-celine-v2test.csv
